<a href="https://colab.research.google.com/github/sreent/data-management-intro/blob/main/Property%20Graphs/10-lab-property-graphs-cypher-traversal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 10 — Property Graphs: Cypher Traversal on a Financial Transfer Network

This lab is **code-based** — you write Cypher queries against a fraud-detection graph loaded into Neo4j running on Google Colab. Most queries use the `%%cypher` magic command from `cellspell`; queries requiring programmatic post-processing (timing measurements, deduplication) use a Python driver `run()` helper.

Use the formal vocabulary (property graph, node, edge/relationship, label, property map, variable-length path, bounded traversal, cycle, supernode, relationship uniqueness) throughout your work.

**Verify-after-predict discipline.** After every query, compare the actual result count to your prediction. If the count does not match, trace back through the Algorithm's Steps 1–3 to find where the mismatch occurred. Record the discrepancy and the diagnosis in your notebook — mismatches are learning opportunities, not failures.

**Materials required:** Google Colab notebook. The seed data file (`seed-data.cypher`) is loaded in the setup cells. No external accounts or cloud setup needed — Neo4j runs locally in Colab.

**Time budget:** Part 1 ~15 min | Part 2 ~15 min | Part 3 ~20 min | Part 4 ~15 min | Part 5 ~15 min | Part 6 ~20 min | Part 7 ~10 min | **Total ~110 min**

---

## Setup

Install Neo4j, the Cypher magic, and load the fraud network dataset.

In [ ]:
# ── Cell 1: Install Neo4j server and Java runtime ─────────
!apt-get update -qq && apt-get install -y -qq openjdk-17-jdk > /dev/null 2>&1
!wget -q https://dist.neo4j.org/neo4j-community-5.26.0-unix.tar.gz
!tar -xf neo4j-community-5.26.0-unix.tar.gz
!neo4j-community-5.26.0/bin/neo4j-admin dbms set-initial-password test1234
!nohup neo4j-community-5.26.0/bin/neo4j console > /dev/null 2>&1 &

In [ ]:
# ── Cell 2: Install cellspell Cypher magic and Neo4j Python driver ─
!pip install "cellspell[cypher] @ git+https://github.com/sreent/jupyter-query-magics.git" neo4j -q

In [ ]:
# ── Cell 3: Wait for Neo4j to become available ────────────
import time, urllib.request
print("Waiting for Neo4j to start...", end="")
for _ in range(30):
    try:
        urllib.request.urlopen("http://localhost:7474", timeout=2)
        print(" Ready!")
        break
    except Exception:
        print(".", end="", flush=True)
        time.sleep(2)
else:
    print(" Timeout — Neo4j may not be running.")

In [ ]:
# ── Cell 4: Load Cypher magic and connect ─────────────────
%load_ext cellspell.cypher
%cypher bolt://localhost:7687 -u neo4j -p test1234

In [ ]:
# ── Cell 5: Download and load the fraud network dataset ───
import urllib.request, os

SEED_URL = "https://raw.githubusercontent.com/sreent/data-management-intro/main/resources/financial-transfers/seed-data.cypher"
SEED_FILE = "seed-data.cypher"

if not os.path.exists(SEED_FILE):
    urllib.request.urlretrieve(SEED_URL, SEED_FILE)
    print(f"Downloaded {SEED_FILE} ({os.path.getsize(SEED_FILE):,} bytes)")

# Read and execute the seed data
with open(SEED_FILE) as f:
    seed_cypher = f.read()

# Strip comment lines and execute
lines = [l for l in seed_cypher.split('\n') if not l.strip().startswith('//')]
clean_query = '\n'.join(lines).strip()
# Remove trailing semicolon if present
if clean_query.endswith(';'):
    clean_query = clean_query[:-1]

from neo4j import GraphDatabase
driver = GraphDatabase.driver("bolt://localhost:7687", auth=("neo4j", "test1234"))
with driver.session() as session:
    session.run(clean_query)
print("Seed data loaded.")

In [ ]:
# ── Cell 6: Python driver helper for programmatic queries ─
# Used from Part 3 onward for timing and deduplication.

def run(query, **params):
    """Execute a Cypher query and return results as a list of dicts."""
    with driver.session() as session:
        result = session.run(query, **params)
        return [record.data() for record in result]

In [ ]:
%%cypher
// Cell 7a: Count nodes
MATCH (n:Account) RETURN count(n) AS node_count

In [ ]:
%%cypher
// Cell 7b: Count relationships
MATCH ()-[r:TRANSFERRED_TO]->() RETURN count(r) AS edge_count


**Expected:** 60 nodes, 150 relationships. Check both counts against these values before continuing. If either count is wrong, re-run Cell 5.

**`_cypher` variable.** After any `%%cypher` cell, the result is also available as the `_cypher` variable (a list of dicts). This is useful for quick Python follow-ups:

In [ ]:
# Previous cell ran a %%cypher query — results are in _cypher
for row in _cypher:
    print(row)


The `run()` helper from Cell 6 is available for queries that need explicit Python post-processing (timing in Q6 and Q9, deduplication in Q7). Use `%%cypher` for standard queries and `run()` when you need to process results programmatically.

**Graph schema:**

```
(:Account {id, name, type, flagged})
    -[:TRANSFERRED_TO {amount, date, transfer_type}]->
(:Account)
```

Account types: Personal, Business, Merchant, Processor.

**Two data scales.** The narrative chapter hand-traces a small subgraph (8 accounts, 9 edges) — the core transfers from Section 3 of the seed data. The full dataset you just loaded (60 accounts, 150 transfers) embeds that subgraph plus surrounding network structure, a planted supernode, and two fraud rings. When a lab query matches the worked example from the chapter, your results will include the hand-traced paths plus additional paths through the wider network.

---

## Part 1 — Load and Inspect

### Q1. Inspect the flagged account

Run the following queries to inspect acct_2049 and one of its outgoing transfers:

In [ ]:
%%cypher
MATCH (a:Account {id: 'acct_2049'}) RETURN a

In [ ]:
%%cypher
MATCH (a:Account {id: 'acct_2049'})-[r:TRANSFERRED_TO]->(b)
RETURN a.id, b.id, r.amount, r.date, r.transfer_type
LIMIT 1


**(a)** Confirm: the amount and date live on the relationship `r`, not on either node. Which properties does node `a` carry? Which does relationship `r` carry?

**(b)** This is Chapter 2's associative entity pattern — OrderLine carried `quantity` and `unit_price` because those values describe the Order–Product relationship, not either entity. Here, `amount`, `date`, and `transfer_type` describe the transfer, not either account. Write one sentence explaining why `amount` cannot live on node `a` or node `b`.

<details><summary>Solution — Q1</summary>

**(a)** Node `a` carries: `id`, `name`, `type`, `flagged`. Relationship `r` carries: `amount`, `date`, `transfer_type`. The relationship data describes the transfer event, not either endpoint.

**(b)** The amount cannot live on node `a` because `a` sends different amounts to different recipients; it cannot live on node `b` because `b` receives different amounts from different senders. The amount describes the specific transfer relationship between `a` and `b`, not either account in isolation — the same reasoning as Chapter 2's OrderLine, where `quantity` describes the Order–Product relationship.

</details>

### Q2. Find the supernode

Find the node with the highest in-degree:

In [ ]:
%%cypher
MATCH (n:Account)<-[r:TRANSFERRED_TO]-()
RETURN n.id, n.name, n.type, count(r) AS in_degree
ORDER BY in_degree DESC
LIMIT 1


**(a)** What is its type? How many incoming edges does it have?

**(b)** Now find its out-degree:

In [ ]:
%%cypher
MATCH (n:Account {id: 'processor_001'})-[r:TRANSFERRED_TO]->()
RETURN count(r) AS out_degree


**(c)** Record the total degree (in-degree + out-degree). You will need this number for cost estimates in Q4, Q6, and Q9.

<details><summary>Solution — Q2</summary>

**(a)** processor_001 (QuickPay Hub), type: Processor. In-degree: 19.

**(b)** Out-degree: 23.

**(c)** Total degree: 19 + 23 = 42. This is the supernode — any traversal that passes through it without edge filtering will expand 42 edges per hop.

</details>

---

## Part 2 — Direct Neighbours (1-hop)

### Q3. Explore the neighbourhood of acct_2049

**(a)** Find all accounts that received a transfer from `acct_2049`:

In [ ]:
%%cypher
MATCH (a:Account {id: 'acct_2049'})-[r:TRANSFERRED_TO]->(b)
RETURN b.id, b.name, r.amount, r.date
ORDER BY r.amount DESC


How many results? Note the out-degree — you will use it to estimate d^k in Q4.

**(b)** Now filter: keep only transfers over $5,000 in the last 90 days (dates after 2024-01-02):

In [ ]:
%%cypher
MATCH (a:Account {id: 'acct_2049'})-[r:TRANSFERRED_TO]->(b)
WHERE r.amount > 5000 AND r.date > date('2024-01-02')
RETURN b.id, b.name, r.amount, r.date
ORDER BY r.amount DESC


How many results remain? This is the **effective degree** after edge filtering — the number that matters for traversal cost.

**(c)** Reverse direction — find all accounts that transferred money *to* `acct_2049`:

In [ ]:
%%cypher
MATCH (b)-[r:TRANSFERRED_TO]->(a:Account {id: 'acct_2049'})
RETURN b.id, b.name, r.amount, r.date


Note: the only change is the arrow direction. In SQL, reversing a self-join requires swapping column references. In Cypher, you flip `->` to `<-`. Direction is structural.

<details><summary>Solution — Q3</summary>

**(a)** 5 outgoing transfers (out-degree = 5). In descending order by amount: acct_3017 ($8,000, wire), acct_1004 ($4,200, wire), acct_4522 ($3,000, ACH), acct_1001 ($2,500, ACH), acct_8800 ($750, card).

**(b)** 1 result: acct_3017 ($8,000, 2024-03-01). The other 4 transfers all fail the amount threshold ($4.2K, $3K, $2.5K, $750 are all ≤ $5K). Effective degree: 1. This dramatically reduces traversal cost: from 5³ = 125 candidate paths (unfiltered) to essentially 1 path per hop.

**(c)** 4 incoming transfers: acct_7801 ($7,000, wire — the cycle-closing edge from Ring 1), acct_9012 ($3,100, ACH), acct_1042 ($2,100, card), acct_1001 ($1,200, card).

</details>

---

## Part 3 — Multi-hop Traversal (Bounded Cypher)

### Q4. Variable-length path query

**Before executing**, look at acct_2049's out-degree from Q3(a).

**(a) Predict:** Estimate the maximum number of paths at depth 3 (d^k). Write your estimate before running the query.

**(b)** Run the variable-length path query:

In [ ]:
%%cypher
MATCH p = (start:Account {id: 'acct_2049'})-[:TRANSFERRED_TO*1..3]->(dest)
RETURN dest.id, length(p) AS depth, [n IN nodes(p) | n.id] AS path
ORDER BY depth, dest.id


How many result rows? Compare to your estimate — why is the actual number smaller?

**(c)** Note: the same destination may appear at multiple depths (via different paths). Why does Cypher return these as separate rows? *(Because variable-length patterns enumerate all matching paths, not just unique destinations.)*

<details><summary>Solution — Q4</summary>

**(a)** From Q3(a), d = 5. Maximum paths at depth 3: 5 + 5² + 5³ = 5 + 25 + 125 = 155. The actual count will be smaller because: (1) not all neighbours have outgoing edges, (2) relationship uniqueness prevents reusing the same edge in a single path, and (3) some paths terminate at dead ends.

**(b)** The actual row count is significantly smaller than the 155 upper bound. The gap between prediction and actual count is the point: d^k estimation tells you the *maximum* cost, not the exact result. Real graphs have uneven degree distributions — the estimate is a worst-case bound.

**(c)** Cypher enumerates all *paths*, not unique *destinations*. If account X is reachable via path A→B→X and also via A→C→X, both appear as separate rows. This matters because in fraud detection, *how* money reached an account (the path) is as important as *whether* it reached the account.

</details>

### Q5. The "follow the money" query — per-hop edge filtering

**Before executing**, predict: Q3(b) showed the effective degree after filtering is 1 (only the $8K transfer to acct_3017 passes). Using your d^k reasoning from Q4, how many filtered paths do you expect at depths 1, 2, and 3? *(Review the worked example trace in the chapter narrative for guidance.)*

In [ ]:
%%cypher
MATCH p = (start:Account {id: 'acct_2049'})-[:TRANSFERRED_TO*1..3]->(dest)
WHERE ALL(r IN relationships(p) WHERE r.amount > 5000 AND r.date > date('2024-01-02'))
RETURN dest.id, length(p) AS depth,
       [n IN nodes(p) | n.id] AS path,
       [r IN relationships(p) | r.amount] AS amounts
ORDER BY depth, dest.id


**(a)** How many paths survived? Compare to your prediction and to Q4's unfiltered count.

**(b)** Inspect the `dest.id` column: does the start account `acct_2049` appear as a destination in any result row? If so, you have found a money-flow cycle — money that left the flagged account and returned to it. Do not filter it out; this is a signal, not a bug. Part 4 explores cycle detection systematically.

**(c)** The `ALL(r IN relationships(p) WHERE ...)` clause applies the predicate at every hop. Explain in one sentence: what is the Cypher equivalent of checking each edge in a BFS loop?

<details><summary>Solution — Q5</summary>

**(a)** At minimum, the 3 paths from the narrative's worked example survive: acct_3017 at depth 1 ($8K), acct_7801 at depth 2 ($8K→$12K), and the cycle back to acct_2049 at depth 3 ($8K→$12K→$7K). Additional paths through the wider network may also survive if all their edges pass both filters. The total is dramatically smaller than Q4 because the edge filter prunes entire subtrees: any path where even one edge fails the predicate is eliminated, along with all deeper extensions.

**(b)** If acct_2049 appears as a destination, you have found Fraud Ring 1 (acct_2049 → acct_3017 → acct_7801 → acct_2049). The amounts along this path are $8K, $12K, $7K — all above the $5K threshold. This is a money-flow cycle, not a bug.

**(c)** `ALL(r IN relationships(p) WHERE ...)` is the declarative equivalent of writing `if not edge_predicate(edge): continue` inside a BFS loop. The student states the constraint; the engine applies it at every hop during traversal.

</details>

### Q6. Two experiments — why both bounds matter

**(a)** Remove the edge filter but keep the depth bound:

In [ ]:
%%cypher
MATCH p = (start:Account {id: 'acct_2049'})-[:TRANSFERRED_TO*1..3]->(dest)
RETURN count(p) AS total_paths


Compare the path count to Q5 (which had the edge filter). Without the filter, every path through the supernode is included.

**(b)** Remove *both* bounds and start from the supernode. *(This experiment uses the `run()` helper from Cell 6.)*

In [ ]:
import time
t0 = time.time()
result = run("""
    MATCH p = (start:Account {id: 'processor_001'})-[:TRANSFERRED_TO*]->(dest)
    RETURN count(p) AS total_paths
""")
print(f"Paths: {result[0]['total_paths']}, Time: {time.time()-t0:.3f}s")


**Predict first:** will this complete quickly, slowly, or not at all on our 60-node graph? Run it and observe. On a small graph, Cypher will eventually terminate (it avoids revisiting the same *relationship* in a path, which prevents infinite loops). But the path count may be surprisingly large. *(On a production graph with millions of nodes, this query would time out or exhaust memory.)*

**(c)** Explain in 2–3 sentences: depth bounds and edge predicates serve *different* purposes. What failure mode does each protect against?

<details><summary>Solution — Q6</summary>

**(a)** The unfiltered path count is much larger than Q5 because every transfer from acct_2049 is followed, including the $3K transfer to acct_4522 which leads to processor_001 (42 edges). Without edge filtering, the supernode's subtree explodes.

**(b)** The unbounded query from processor_001 will complete on this small graph (60 nodes) but produces a surprisingly large path count. On a production graph, it would time out.

**(c)** Depth bounds protect against **unbounded exploration** — without `*1..k`, the engine traverses the entire reachable graph, which may be millions of nodes. Edge predicates protect against **irrelevant path proliferation** — without `ALL(... WHERE ...)`, the engine follows every edge including low-value transfers and old transactions, producing an explosion through supernodes. Both are needed: depth bounds limit *how far*, edge predicates limit *which edges*.

</details>

---

## Part 4 — Cycle Detection (Fraud Rings)

### Q7. Find all money-flow rings

**Before executing**, predict: the dataset contains 2 planted fraud rings (lengths 3 and 4). Cypher matches each cycle starting from every node in the ring. How many raw rows do you expect? Write your prediction before running.

In [ ]:
%%cypher
MATCH p = (a:Account)-[:TRANSFERRED_TO*3..6]->(a)
WHERE ALL(r IN relationships(p) WHERE r.amount > 3000)
RETURN [n IN nodes(p) | n.id] AS ring,
       length(p) AS ring_length,
       [r IN relationships(p) | r.amount] AS amounts


**(a)** How many result rows? Compare to your prediction.

**Important:** Cypher matches cycles starting from *every* node in the ring. A 3-node cycle A→B→C→A produces 3 rows: [A,B,C,A], [B,C,A,B], and [C,A,B,C]. These are the *same ring* with different starting points. A 4-node cycle produces 4 rows.

**(b)** To find the number of *unique* rings, deduplicate in Python:

In [ ]:
results = run("""
    MATCH p = (a:Account)-[:TRANSFERRED_TO*3..6]->(a)
    WHERE ALL(r IN relationships(p) WHERE r.amount > 3000)
    RETURN [n IN nodes(p) | n.id] AS ring,
           length(p) AS ring_length,
           [r IN relationships(p) | r.amount] AS amounts
""")

# Deduplicate: normalise each ring by sorting its non-repeated node IDs
seen = set()
unique_rings = []
for row in results:
    key = tuple(sorted(row['ring'][:-1]))  # sort node IDs, drop repeated start
    if key not in seen:
        seen.add(key)
        unique_rings.append(row)

print(f"Total rows: {len(results)}")
print(f"Unique rings: {len(unique_rings)}")
for ring in unique_rings:
    print(f"  Ring: {ring['ring']}, Amounts: {ring['amounts']}")


How many unique rings after deduplication? List the account IDs in each. Compare to the ground truth (the dataset contains 2 planted fraud rings).

<details><summary>Solution — Q7</summary>

**(a)** Ring 1 (length 3) produces 3 raw rows (one per starting node): [acct_2049, acct_3017, acct_7801, acct_2049], [acct_3017, acct_7801, acct_2049, acct_3017], [acct_7801, acct_2049, acct_3017, acct_7801]. Ring 2 (length 4) produces 4 raw rows. Total: 7 raw rows.

**(b)** After deduplication: 2 unique rings.
- Ring 1: acct_2049 → acct_3017 → acct_7801 → acct_2049 (amounts: $8K, $12K, $7K)
- Ring 2: acct_5566 → acct_1234 → acct_5678 → acct_9012 → acct_5566 (amounts: $6K, $5.5K, $4.5K, $4K)

This matches the 2 planted fraud rings in the ground truth.

</details>

### Q8 (bonus). Bottleneck estimation

For one fraud ring from Q7, compute the minimum edge amount around the cycle — the bottleneck that limits how much money could flow through the ring. Write a Cypher query that returns the minimum amount:

In [ ]:
%%cypher
MATCH p = (a:Account)-[:TRANSFERRED_TO*3..6]->(a)
WHERE ALL(r IN relationships(p) WHERE r.amount > 3000)
WITH [n IN nodes(p) | n.id] AS ring,
     reduce(minAmt = head(relationships(p)).amount,
            r IN tail(relationships(p)) |
            CASE WHEN r.amount < minAmt THEN r.amount ELSE minAmt END) AS bottleneck
RETURN ring, bottleneck


*(The `reduce` seeds with the first relationship's amount, then folds over the rest — no magic numbers.)* This is how investigators estimate laundered volume.

<details><summary>Solution — Q8</summary>

Ring 1 bottleneck: min($8K, $12K, $7K) = **$7,000**. This is the maximum amount that could flow through the entire cycle — the smallest edge limits throughput, just as the narrowest pipe limits water flow.

Ring 2 bottleneck: min($6K, $5.5K, $4.5K, $4K) = **$4,000**.

</details>

---

## Part 5 — Supernode Handling

### Q9. Measure the effect of edge filtering on the supernode

**Before executing**, estimate: processor_001 has 23 outgoing edges. At depth 2 without filters, the maximum paths is 23 + 23 × d₂ (where d₂ is each neighbour's out-degree). With filters (amount > $10K, wire only, after 2024-02-01), estimate what fraction of the 23 outgoing edges pass all three conditions, then predict the filtered path count. Write your estimates.

Run a 2-hop traversal from `processor_001` twice — without and with edge filters — and compare:

In [ ]:
# Without filters
t0 = time.time()
result_a = run("""
    MATCH p = (start:Account {id: 'processor_001'})-[:TRANSFERRED_TO*1..2]->(dest)
    RETURN count(p) AS total_paths, count(DISTINCT dest) AS unique_destinations
""")
time_a = time.time() - t0

# With filters: transfers over $10K, wire only, last 60 days
t0 = time.time()
result_b = run("""
    MATCH p = (start:Account {id: 'processor_001'})-[:TRANSFERRED_TO*1..2]->(dest)
    WHERE ALL(r IN relationships(p)
              WHERE r.amount > 10000
                AND r.transfer_type = 'wire'
                AND r.date > date('2024-02-01'))
    RETURN count(p) AS total_paths, count(DISTINCT dest) AS unique_destinations
""")
time_b = time.time() - t0

print(f"Unfiltered: {result_a[0]}, Time: {time_a:.3f}s")
print(f"Filtered:   {result_b[0]}, Time: {time_b:.3f}s")
print(f"Reduction:  {result_a[0]['total_paths'] / max(result_b[0]['total_paths'], 1):.0f}x fewer paths")


**(a)** What is the reduction factor? What is the speedup?

**(b)** This is the graph-traversal equivalent of Chapter 5's EXPLAIN before/after adding an index: measure, then improve, then measure again. Write one sentence explaining why edge predicates are to graph traversal what indexes are to relational queries.

### Q10. Alternative strategy — skip the supernode

Use a node predicate to exclude Processor nodes from intermediate hops:

In [ ]:
%%cypher
MATCH p = (start:Account {id: 'acct_2049'})-[:TRANSFERRED_TO*1..3]->(dest)
WHERE ALL(r IN relationships(p) WHERE r.amount > 5000)
  AND NONE(n IN nodes(p)[1..-1] WHERE n.type = 'Processor')
RETURN dest.id, length(p) AS depth, [n IN nodes(p) | n.id] AS path


**(a)** The `NONE(n IN nodes(p)[1..-1] WHERE ...)` checks intermediate nodes only (not start or end). `[1..-1]` is Cypher list slicing: all elements except the first and last. Compare results to Q5 — which accounts are no longer reachable?

**(b)** Are any of the excluded accounts relevant to the investigation? When would you use a node predicate (skip the supernode) vs an edge predicate (filter on the supernode's edges)?

<details><summary>Solution — Q9–Q10</summary>

**Q9(a):** Unfiltered 2-hop from processor_001: with 23 outgoing edges, hop 1 alone produces 23 paths, and hop 2 fans out further through each neighbour's edges. The filtered query (>$10K, wire, after 2024-02-01) passes only a small subset of processor_001's outgoing edges at hop 1, and each survivor's neighbours must also pass all three conditions at hop 2. The reduction factor is substantial — compare the two path counts to confirm. On this small dataset (60 nodes) the timing difference may be subtle, but the path count reduction demonstrates the layered-filtering principle.

**Q9(b):** Edge predicates reduce the search space *before* traversal expands, just as an index reduces the scan space before row retrieval. Both are pre-filtering mechanisms that avoid examining irrelevant data.

**Q10(a):** With `NONE(... WHERE n.type = 'Processor')`, any path that passes through processor_001 as an intermediate node is rejected. Accounts reachable only via processor_001 (e.g., those on the far side of the supernode) disappear from the results.

**Q10(b):** Node predicates (skip the supernode entirely) are appropriate when: (1) the supernode is a payment processor or aggregator that adds no investigative value, or (2) the traversal cost is too high even with edge filtering. Edge predicates (filter on the supernode's edges) are appropriate when: (1) some transfers through the supernode are relevant, or (2) you need to trace specific high-value paths through the processor.

</details>

---

## Part 6 — Cross-Model Comparison

### Q11. Read and analyse the SQL recursive CTE

Below is a SQL recursive CTE that attempts to answer Q5's question: "from account 2049, follow transfers over $5,000 within 3 hops in the last 90 days." Read it carefully, then answer the questions below.

**SQL recursive CTE for comparison (read-only — not executable in this notebook):**

```sql
%%sql
WITH RECURSIVE money_trail AS (
  SELECT t.to_id, t.amount, t.txn_date, 1 AS depth,
         CAST(t.from_id AS CHAR(500)) AS visited_path
  FROM Transfer t
  WHERE t.from_id = 'acct_2049'
    AND t.amount > 5000
    AND t.txn_date > DATE_SUB(NOW(), INTERVAL 90 DAY)

  UNION ALL

  SELECT t.to_id, t.amount, t.txn_date, mt.depth + 1,
         CONCAT(mt.visited_path, ',', t.from_id)
  FROM Transfer t
  JOIN money_trail mt ON t.from_id = mt.to_id
  WHERE mt.depth < 3
    AND t.amount > 5000
    AND t.txn_date > DATE_SUB(NOW(), INTERVAL 90 DAY)
    AND FIND_IN_SET(t.to_id, mt.visited_path) = 0
)
SELECT DISTINCT to_id, depth FROM money_trail ORDER BY depth;
```



**(a)** Identify the cycle-prevention mechanism — what data type is it using, and what happens if an account ID contains a comma?

**(b)** The edge filter (`amount > 5000`, `txn_date > ...`) appears in both the base case and the recursive case. Why must it be duplicated?

**(c)** Write the Cypher equivalent (Q5) side-by-side. Rate the readability of each (1–5) for a compliance officer who needs to audit the logic.

### Q12. Read and complete the SPARQL reified query

Below is a partial SPARQL query that attempts to answer Q5's question using reified transfers. Complete the missing Hop 3 block, then answer the question.

```sparql
PREFIX ex: <http://example.org/fraud#>
SELECT ?hop1 ?hop2 ?hop3 WHERE {
  # Hop 1
  ?txn1 a ex:Transfer ;
        ex:fromAccount ex:acct_2049 ; ex:toAccount ?hop1 ;
        ex:amount ?a1 ; ex:txnDate ?d1 .
  FILTER (?a1 > 5000 && ?d1 > "2024-01-01"^^xsd:date)

  # Hop 2
  ?txn2 a ex:Transfer ;
        ex:fromAccount ?hop1 ; ex:toAccount ?hop2 ;
        ex:amount ?a2 ; ex:txnDate ?d2 .
  FILTER (?a2 > 5000 && ?d2 > "2024-01-01"^^xsd:date)

  # Hop 3 — complete this block
  ___
}
```

**(a)** Complete Hop 3, then add cycle-prevention FILTERs so that no account appears twice in the path. How many pairwise inequality checks are needed for 3 hops? Derive the general formula for k hops and explain why the count grows quadratically.

**(b)** If the compliance team asks for 5 hops instead of 3, how many lines do you add in SPARQL? How does the Cypher query change? *(This closes the Chapter 9 property path bridge.)*

<details><summary>Solution — Q11–Q12</summary>

**Q11(a):** The cycle-prevention mechanism uses `CAST(... AS CHAR(500))` and `CONCAT` to build a comma-separated string of visited account IDs, then `FIND_IN_SET` to check membership. If an account ID contains a comma (e.g., `acct_1,000`), `FIND_IN_SET` splits incorrectly and the check produces false positives or negatives. This is a string-manipulation hack, not a structural solution.

**Q11(b):** The edge filter must be duplicated because SQL recursive CTEs have separate base-case and recursive-case SELECT statements. The base case runs once (depth 1); the recursive case runs for each subsequent depth. A filter omitted from either half would allow unfiltered rows at that depth.

**Q11(c):** Readability: Cypher 5/5 (reads like the English requirement), SQL 2/5 (control flow hidden in UNION ALL, string hacks), SPARQL 1/5 (each hop requires 6 lines of boilerplate).

**Q12(a):** Hop 3 is another 6-line block (copy of Hop 2 with `?hop2` → `?hop3`, `?txn3`, `?a3`, `?d3`). For cycle prevention, add pairwise FILTERs: `FILTER(?hop1 != ?hop2 && ?hop1 != ?hop3 && ?hop2 != ?hop3)`. That's C(3,2) = 3 pairwise checks. General formula: C(k,2) = k(k-1)/2 checks for k hops. This grows quadratically because each new hop must be compared to all previous hops.

**Q12(b):** SPARQL: add 2 more 6-line blocks (Hops 4 and 5) + additional FILTER pairs: 12 lines + C(5,2)=10 inequality checks. Cypher: change `*1..3` to `*1..5` — one integer.

</details>

### Q13. Fill in the comparison table

| Criterion | SQL (recursive CTE) | SPARQL (reified) | Cypher |
|---|---|---|---|
| Lines of code for Q5 | | | |
| Query paradigm | | | |
| Cycle prevention | | | |
| Per-hop edge filtering | | | |
| Readability (1–5) | | | |
| Changing from 3 hops to 5 hops | | | |
| When would you choose this model? | | | |

<details><summary>Solution — Q13</summary>

| Criterion | SQL (recursive CTE) | SPARQL (reified) | Cypher |
|---|---|---|---|
| Lines of code for Q5 | ~20 | ~25+ (without cycle prevention) | ~6 |
| Query paradigm | Declarative (SQL) with imperative depth tracking | Declarative (pattern matching) with manual hop unrolling | Declarative (pattern matching) with built-in traversal |
| Cycle prevention | String-concatenation hack (CAST/CONCAT/FIND_IN_SET) — fragile | Manual FILTER with O(k²) inequality checks | Automatic — relationship uniqueness is built in |
| Per-hop edge filtering | Duplicated in base case and recursive case | One FILTER per hop block (duplicated per hop) | Single `ALL(r IN relationships(p) WHERE ...)` clause |
| Readability (1–5) | 2 — opaque control flow, string hacks | 1 — boilerplate explosion, hardcoded hops | 5 — reads like the English requirement |
| Changing from 3 hops to 5 hops | Add recursive case logic, extend string capacity | Copy-paste two 6-line blocks + add inequality checks | Change one integer: `*1..5` |
| When would you choose this model? | Regulatory reporting with strict constraints, complex aggregation across normalised tables | Integrating data from multiple sources with different schemas; identity resolution via URIs | Traversal-heavy workloads: fraud detection, network analysis, recommendation engines |

</details>

---

## Part 7 — Reflection

### Q14. Four-model synthesis

Looking across all chapters: relational (Chapters 2–5), document/JSON/XML (Chapters 6–7), RDF (Chapters 8–9), property graph (Chapter 10). For each model, name one question it answers well and one it resists. Address all four models:

**(a)** *Relational vs property graph (anchor).* The relational schema from Chapters 2–5 records transactions correctly (normalised, FK-constrained, with CHECK constraints and transaction isolation). The property graph in this lab detects traversal patterns across those transactions. Why does the relational model resist the fraud-ring query, and why would a property graph resist the normalised, multi-table insert pattern with FK constraints and CHECK constraints that Chapters 2–5 rely on? Reference a specific design feature from Chapters 2–5 (e.g., normalisation, FK constraints, transaction isolation) that creates the resistance. *(Note: Neo4j supports ACID transactions — the resistance is not about atomicity, but about the relational design patterns of normalised schemas with declarative constraints across multiple tables.)*

**(b)** *Document.* Could you embed transfer history inside account documents? What happens when a transfer connects two accounts in different documents? (This is the cross-document reference problem from Chapter 6.)

**(c)** *RDF.* What does integration across sources (Chapter 8) give you? What does the reification cost from Q12 take away?

**(d)** *Synthesis.* Which model would you reach for first if the primary workload were: (i) regulatory reporting with strict constraints, (ii) fraud-ring detection, (iii) merging data from three different banks? One sentence each is enough.

This is preparation for Chapter 11's capstone comparison. Write ~½ page total across (a)–(d).

<details><summary>Solution — Q14</summary>

**(a)** *Relational vs property graph.* The relational model resists the fraud-ring query because multi-hop traversal requires recursive CTEs with manual depth tracking and string-hack cycle prevention — the relational model treats relationships as foreign keys (pointers) or junction table rows (separate entities requiring joins), not as traversable, data-carrying edges. The property graph model would resist the normalised, multi-table insert pattern from Chapters 2–5 because its strength is traversal, not constraint enforcement across multiple interrelated entities with FK constraints, CHECK constraints, and transaction isolation enforcing business rules across normalised tables.

**(b)** *Document.* Embedding transfer history inside account documents would mean each transfer appears in two documents (the sender's and the receiver's) — a cross-document reference problem (Chapter 6). Updates must be synchronised across both documents. When the primary query is "follow transfers across accounts," the embedded design forces repeated lookups across documents — the traversal crosses document boundaries at every hop.

**(c)** *RDF.* Integration across sources (Chapter 8) is RDF's strength: if three banks publish transfer data using shared URIs, the graphs merge automatically by URI alignment — no ETL pipeline needed. The reification cost (Q12) takes away readability and query simplicity: each transfer requires 6 triples, each hop requires 6 triple patterns, and cycle prevention requires O(k²) inequality checks.

**(d)** *Synthesis.* (i) Regulatory reporting with strict constraints → relational model (normalised schema with FK constraints, CHECK constraints, transaction isolation). (ii) Fraud-ring detection → property graph model (Cypher traversal with per-hop filtering, native cycle detection). (iii) Merging data from three banks → RDF (URI-based identity, automatic integration by URI alignment).

</details>

---

## Deliverable

- Colab notebook with Neo4j setup and Q1–Q14 answers (Cypher queries + written responses). Q8 is optional/bonus.
- Comparison table (Q13) completed.
- Reflection (Q14): ~½ page.
- **Carries forward to Chapter 11:** The comparison tables from Q13 and Q14 are direct input to the capstone exercise. Students should keep them accessible.